![digitizing_team](digitizing_team.png)


DigiNsure Inc. is an innovative insurance company focused on enhancing the efficiency of processing [claims](https://www.investopedia.com/terms/i/insurance_claim.asp) and customer service interactions. Their newest initiative is digitizing all historical insurance claim documents, which includes improving the labeling of some IDs scanned from paper documents and identifying them as primary or secondary IDs.

To help them in their effort, you'll be using multi-modal learning to train an Optical Character Recognition (OCR) model. To improve the classification, the model will use **images** of the scanned documents as input and their **insurance type** (home, life, auto, health, or other). Integrating different data modalities (such as image and text) enables the model to perform better in complex scenarios, helping to capture more nuanced information. The **labels** that the model will be trained to identify are of two types: a primary and a secondary ID, for each image-insurance type pair.

If you are using [Google Colab](https://colab.google/)/[Jupyter lite](https://jupyter.org/try-jupyter/lab/), run the following cell to install the required packages. If you are using a virtual environment, make sure to install the packages in that environment. You can do this by running the command `pip install -r requirements.txt` in your terminal. The `requirements.txt` file is included in the repository.

```bash
#update the pip version
pip install --upgrade pip
# make virtual environment
python3 -m venv .venv
#install the required packages
pip install -r requirements.txt
```
```bash
# activate the virtual environment
source .venv/bin/activate
# or on Windows
.venv\Scripts\activate
```

Make sure you open jupyter with the kernel of the virtual environment. You can do this by running the command `jupyter notebook` or `jupyter lab` in your terminal after activating the virtual environment. This will ensure that the packages installed in the virtual environment are available in Jupyter.

```bash
# install jupyter in the virtual environment
pip install jupyter
# install jupyter lab in the virtual environment
pip install jupyterlab
# run jupyter notebook
jupyter notebook
# run jupyter lab
jupyter lab
```


In [5]:
# !pip install -q torch
# !pip install -q torchvision
# !pip install -q matplotlib
# !pip install -q tabulate

Current computer specifications:

In [ ]:
# OS information
!uname -srm
!cat /etc/os-release

In [36]:
!lscpu | grep "Model name"
!lscpu | grep "CPU(s):"
!lscpu | grep "Thread(s) per core"
!lscpu | grep "Core(s) per socket"

In [37]:
!free -h | grep Mem

In [38]:
!df -h /

In [39]:
!lspci | grep VGA

In [ ]:
# Import the necessary libraries
import os  # Provides functions to interact with the operating system
import random  # Implements pseudo-random number generators for various distributions
from project_utils import ProjectDataset  # Custom dataset class for handling project-specific data
from collections import Counter  # Provides a dictionary subclass for counting hashable objects
import pandas as pd  # Data manipulation and analysis library
from tabulate import tabulate  # Pretty-print tabular data
import matplotlib.pyplot as plt  # Plotting library for creating static, animated, and interactive visualizations
import numpy as np  # Fundamental package for scientific computing with Python
import torch  # PyTorch library for tensor computation and deep learning
import torch.nn as nn  # Provides neural network layers and functions
from torch.utils.data import DataLoader  # Utility functions for loading data in batches
from torchvision import transforms  # Common image transformations for computer vision
import torch.optim as optim  # Optimization algorithms for training neural networks
import torch.nn.functional as F  # Provides functions for various neural network operations
from torch.optim.lr_scheduler import OneCycleLR  # Learning rate scheduler for training
from torch.utils.data import DataLoader, random_split  # Utility functions for loading and splitting data
import pickle  # Serializing and de-serializing Python object structures
from tqdm import tqdm  # Progress bar library for loops and iterations
import optuna  # Hyperparameter optimization library

In [7]:
# Load the data from the pickle file
# Sometimes these files maybe marked as unsafe especially in huggingface hub
dataset = pickle.load(open('ocr_insurance_dataset.pkl', 'rb'))

# Define a function to visualize codes with their corresponding types and labels
def show_dataset_images(dataset, num_images=5):
    '''
    A function to visualize codes with their corresponding types and labels

    Parameters
    ----------
    dataset : ProjectDataset
        The dataset to visualize.

    num_images : int
        The number of images to visualize.

    Returns
    -------
    None

    Examples
    --------
    >>> show_dataset_images(dataset, num_images=5)
    '''
    fig, axes = plt.subplots(1, min(num_images, len(dataset)), figsize=(20, 4))
    for ax, idx in zip(axes, np.random.choice(len(dataset), min(num_images, len(dataset)), False)):
        img, lbl = dataset[idx]
        ax.imshow((img[0].numpy() * 255).astype(np.uint8).reshape(64,64), cmap='gray'), ax.axis('off')
        ax.set_title(f"Type: {list(dataset.type_mapping.keys())[img[1].tolist().index(1)]}\nLabel: {list(dataset.label_mapping.keys())[list(dataset.label_mapping.values()).index(lbl)]}")
    plt.show()

# Inspect 5 codes images from the dataset
show_dataset_images(dataset)

The image shows several examples of scanned documents containing text, each labeled with an insurance type and a corresponding ID label. The text in the images is not always clear and appears to be somewhat blurred, which can pose a challenge for OCR (Optical Character Recognition) tasks. The goal of the project is to train a model that can accurately classify these images based on their insurance types (e.g., home, health, auto, life) and their corresponding ID labels (e.g., primary_id, secondary_id).

Next we check the dimensionality of the dataset. That is, check the number of samples, the number of classes, and the data types of the labels that is, home, health, auto, life, and other. The dataset is a collection of images and their corresponding labels. Then we'll pick a single sample and validate how to extract it from a tuple. The dataset is a list of tuples, where each tuple contains an image and its corresponding label. Then we'll count the number of samples and categories they represent to confirm if we have a balanced dataset. 


In [8]:
# Checking dimensions of the dataset
print(f"Number of samples in the dataset: {len(dataset)}")
print(f"Number of classes in the dataset: {len(dataset.label_mapping)}")
print(f"Names of the classes in the dataset: {list(dataset.label_mapping.keys())}")
print(f"Number of types in the dataset: {len(dataset.type_mapping)}")
print(f"Names of the types in the dataset: {list(dataset.type_mapping.keys())}")

In [9]:
# grab the first image and its corresponding type vector and label
# confirm the dimensions of the image, type vector, and label and their types
(image, type_vec), label = dataset[0]
print(f"Image shape: {image.shape}, Image type: {image.dtype}")
print(f"Type vector shape: {type_vec.shape}, Type vector type: {type_vec.dtype}")
print(f"Label: {label}")

In [ ]:
# go through a sample of 10 images and their corresponding type vectors and labels
# skip the first item by changing the range to start from 1
# this is important to view especially if there are issues with any of the images, type vectors, or labels
for i in range(1, 11):
    (image, type_vec), label = dataset[i]
    print(f"Image shape: {image.shape}, Image type: {image.dtype}")
    print(f"Type vector shape: {type_vec.shape}, Type vector type: {type_vec.dtype}")
    print(f"Label: {label}")
    print()


In [12]:
def count_samples(dataset):
    """
    Counts the number of samples per class and type in the dataset.

    Parameters
    ----------
    dataset : ProjectDataset
        The dataset instance to analyze.

    Returns
    -------
    None

    Examples
    --------
    >>> count_samples(dataset)
    """
    # Count labels
    label_counts = Counter(dataset.labels)
    print("Class Counts:")
    for label, count in label_counts.items():
        label_name = [k for k, v in dataset.label_mapping.items() if v == label][0]
        print(f"{label_name}: {count}")

    # Count types
    type_indices = [sample[1].argmax().item() for sample in dataset.data]  # Assuming one-hot encoding
    type_counts = Counter(type_indices)
    print("\nType Counts:")
    for type_name, index in dataset.type_mapping.items():
        print(f"{type_name}: {type_counts.get(index, 0)}")

In [13]:
count_samples(dataset)

The number of samples in the dataset is 100. The distribution of the insurance types is as follows:

- Home: 16 samples
- Life: 20 samples
- Auto: 26 samples
- Health: 16 samples
- Other: 22 samples

The class distribution is balanced, and the dataset is small. However, if you initialize the class with more samples and data augmentation, could help increase the number of samples and improve the model's performance. Surprisingly, this didn't work for some of my experiments. I may need to review what kind of practices that are used to augment this kind of data.

## Model development

The dataset has a combination of text and image features that we need sort in a way a neural network can make sense of the information. We'll develop an Optical Character Recognition model to sort categories of ID codes extracted from scanned insurance documents. 

The next step is to create a model. We can try this specification:

    * A network architecture with different layers to ingest both the image and the type. The layers relative to the image should be saved as a sequential module called image_layer, and compatible the input dimensions of the images (64x64 pixels size).

    * A Conv2d layer with kernel size 3x3, padding 1, and appropriate in/out channels.

Train the Model: call the model as model and use an appropriate optimizer and loss function to train it. Iterate through your training for ten epochs.

What I can glean from this description is that the model should be a multi-modal model that takes both image and text as input. The image input should be processed through a convolutional neural network (CNN) to extract features, while the text input should be processed through an embedding layer to convert the text into a numerical representation. The two outputs should then be concatenated and passed through a fully connected layer to produce the final output.

## Normalize images

Why? So that we can meetthe input requirements of the model. The images are normalized to have pixel values between 0 and 1, which is a common practice in deep learning. This helps the model learn better and faster. The normalization process involves dividing each pixel value by 255, which is the maximum pixel value for an 8-bit image. This ensures that all pixel values are in the range [0, 1].

In [14]:
def get_dataset_stats(dataset):
    """
    Calculate mean and std of image dataset where each item is (image, type_data) tuple

    Parameters
    ----------
    dataset : ProjectDataset
        The dataset instance to analyze. that contains (image, type_data) tuples

    Returns
    -------
    mean : float
        The mean value of the dataset images

    std : float
        The standard deviation of the dataset images

    Examples
    --------
    >>> get_dataset_stats(dataset)
    (0.5, 0.5)
    """
    # Extract just the images from tuples and convert to tensors
    images = torch.stack([torch.tensor(img[0]) if isinstance(img[0], np.ndarray)
                         else img[0] for img in dataset.data])

    # Calculate statistics
    mean = images.mean()
    std = images.std()
    return mean, std

# Calculate dataset statistics
mean, std = get_dataset_stats(dataset)

# Define normalization with computed statistics
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[mean.item()], std=[std.item()])
])

Apart from finding the mean and standard deviation. We are going to working with randomly initialized  weights. We want you to get the results we got from our experiments. Therefore, we need to set random seeds for all the libraries we are using. This will ensure that the results are reproducible and consistent across different runs. The random seeds are set for the following libraries: numpy, torch, and random. The random seed is set to 42, which is a common practice in machine learning to ensure reproducibility.

In [16]:
def set_seeds(seed=42):
    """
    Set seeds for reproducibility.

    Parameters
    ----------
    seed : int
        The seed value to use (default: 42)
    """
    # Python random
    random.seed(seed)

    # Numpy
    np.random.seed(seed)

    # PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # for multi-GPU

    # PyTorch backends
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # Set a fixed value for the hash seed
    os.environ["PYTHONHASHSEED"] = str(seed)

    print(f"Random seed set to {seed}")

# Call this before creating your model and running training
set_seeds(256)

Now, we will make validation splits to ensure that our model is able to generalize. This is the core metric when it comes to classification tasks. We will split the dataset into training and validation sets. The training set will be used to train the model, while the validation set will be used to evaluate the model's performance. The split is done in a stratified manner to ensure that the class distribution is maintained in both sets. This is important because we want to ensure that the model is able to generalize well to unseen data.

In [19]:
# Split the dataset into training, validation, and test sets
train_size = int(0.8 * len(dataset))
valid_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - valid_size
train_dataset, valid_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, valid_size, test_size]
)

# Set the transform for each split (if your original dataset supports a transform attribute)
train_dataset.dataset.transform = transform
valid_dataset.dataset.transform = transform
# Optionally, if you want to apply the same normalization for the test set:
test_dataset.dataset.transform = transform

# Define stronger augmentations
train_transforms = transforms.Compose([
    # Random affine: rotation, translation, scaling
    transforms.RandomAffine(degrees=5, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    # Random perspective: distortion of the image
    transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
    # remove random pixels in the image
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
    # Scale the image to a fixed size
    transforms.Normalize((0.5,), (0.5,))  # Normalize to [-1, 1]
])

# Apply transforms to your dataset loading code
train_dataset.dataset.transform = train_transforms
valid_dataset.dataset.transform = transform
test_dataset.dataset.transform = transform

# Create data loaders with the new transformations
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True)
valid_loader = DataLoader(valid_dataset, batch_size=18, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=18, shuffle=False)

In [20]:
class ResidualBlock(nn.Module):
    """
    A standard residual block with two convolutional layers and a skip connection
    """
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()

        # First convolutional layer
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                              stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)

        # Second convolutional layer
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                              stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Skip connection (identity mapping or 1x1 conv if dimensions change)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                         stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        # Main path
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        # Skip connection
        out += self.shortcut(x)

        # Final activation
        out = F.relu(out)

        return out

In [21]:
class OCRModel(nn.Module):
    def __init__(self, num_classes=2, num_types=5, dropout1=0.3, dropout2=0.2):
        super().__init__()

        # Image processing path with residual connections
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(256)

        # Add residual blocks
        self.res1 = ResidualBlock(64, 64)
        self.res2 = ResidualBlock(128, 128)

        # Type embedding
        self.type_embedding = nn.Embedding(num_types, 32)
        self.type_linear = nn.Linear(32, 128)

        # Save dropout rates
        self.dropout1 = dropout1
        self.dropout2 = dropout2

        # Combined processing
        self.fc = nn.Sequential(
            nn.Linear(256 * 8 * 8 + 128, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout1),  # Use the parameter here
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout2),  # Use the parameter here
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # Unpack the input (image, type_vector)
        image, type_vec = x

        # Image processing branch with residual connections
        x1 = F.relu(self.bn1(self.conv1(image)))
        x1 = self.res1(x1)
        x1 = F.max_pool2d(x1, 2)

        x1 = F.relu(self.bn2(self.conv2(x1)))
        x1 = self.res2(x1)
        x1 = F.max_pool2d(x1, 2)

        x1 = F.relu(self.bn3(self.conv3(x1)))
        x1 = F.max_pool2d(x1, 2)
        x1 = torch.flatten(x1, 1)

        # Type processing
        # Find the index of the 1 in the one-hot encoded type_vec
        type_idx = torch.argmax(type_vec, dim=1)
        x2 = self.type_embedding(type_idx)
        x2 = F.relu(self.type_linear(x2))

        # Combine features
        combined = torch.cat((x1, x2), dim=1)

        # Final classification
        return self.fc(combined)

In [22]:
# Show the model architecture and no of parameters
model = OCRModel(num_classes=len(dataset.label_mapping), num_types=len(dataset.type_mapping))
print(model)

def count_parameters(model):
    """
    Count the number of parameters in a PyTorch model.

    Parameters
    ----------
    model : nn.Module
        The PyTorch model instance.

    Returns
    -------
    int
        The total number of parameters in the model.
    """
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Count the number of parameters in the model
num_params = count_parameters(model)
print(f"Number of parameters in the model: {num_params}")


In [ ]:
# show how the training data looks like in terms of matries and scalars
# one example for single batch
for batch in train_loader:
    images, labels = batch[0], batch[1]
    break

# see some data
images[0], labels[0]

# mermaid diagram of the inner workings of the next steps
```mermaid
graph LR
    A[Import Libraries] --> B(EarlyStopping Class);
    A --> C(analyze_class_distribution Function);
    A --> D(get_class_weights Function);
    A --> E(to_device Function);
    A --> F(train_model Function);
    A --> G(objective Function);
    A --> H(run_hyperparameter_search Function);

    B --> I{Initialize EarlyStopping};
    I --> J{Check Validation Loss};
    J -- Improve --> K{Reset Counter, Save Best Score};
    J -- No Improve --> L{Increment Counter};
    L --> M{Counter >= Patience?};
    M -- Yes --> N{Set Early Stop};
    M -- No --> O{Continue Training};

    C --> P{Iterate Through Loader};
    P --> Q{Extract Labels};
    Q --> R{Count Class Occurrences};

    D --> S{Call analyze_class_distribution};
    S --> T{Calculate Class Weights};

    E --> U{Check Data Type};
    U -- Tensor --> V{Move to Device};
    U -- List/Tuple --> W{Recursively Move Elements};
    U -- Other --> X{Attempt Move to Device};

    F --> Y{Set Device, Debug Data};
    Y --> Z{Analyze Class Distribution};
    Z --> AA{Define Loss, Optimizer, Scheduler};
    AA --> BB{Initialize EarlyStopping};
    BB --> CC{Training Loop};
    CC --> DD{Forward, Backward, Optimize};
    CC --> EE{Validate};
    EE --> FF{Check Early Stopping};
    FF --> GG{Load Best Model};
    GG --> HH{Visualize Results};

    G --> II{Suggest Hyperparameters};
    II --> JJ{Create DataLoaders};
    JJ --> KK{Create Model};
    KK --> LL{Train Model};
    LL --> MM{Return Validation Loss};

    H --> NN{Define Objective Wrapper};
    NN --> OO{Create Optuna Study};
    OO --> PP{Optimize Study};
    PP --> QQ{Return Study, Best Params};
    QQ --> RR{Visualize Study Results};
```

In [ ]:

# Early stopping implementation
class EarlyStopping:
    '''Once certain parameters are met, stop training the model: patience'''
    def __init__(self, patience=7, min_delta=0, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.verbose = verbose

    def __call__(self, val_loss):
        if self.best_score is None:
            self.best_score = val_loss
        elif val_loss > self.best_score + self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = val_loss
            self.counter = 0

def analyze_class_distribution(loader):
    '''Review the class distribution in the dataset

    Parameters
    ----------
    loader : DataLoader
        The DataLoader instance to analyze.

    Returns
    -------
        dict
        A dictionary with class labels as keys and their counts as values.

    '''
    class_counts = {}
    for batch in loader:
        _, labels = batch
        if isinstance(labels, torch.Tensor):
            for label in labels.cpu().numpy():
                if label not in class_counts:
                    class_counts[label] = 0
                class_counts[label] += 1
    return class_counts

def get_class_weights(train_loader):
    '''Get class weights for the training data

    Parameters
    ----------
    train_loader : DataLoader
        The DataLoader instance for training data.

    Returns
    -------
        torch.Tensor
        A tensor containing the class weights.
    '''
    class_counts = analyze_class_distribution(train_loader)
    print("Class distribution:", class_counts)

    if not class_counts:  # Check if class_counts is empty
        print("Warning: No valid labels found in the data loader!")
        return torch.ones(2)  # Return default weights

    total = sum(class_counts.values())
    weights = {cls: total/count for cls, count in class_counts.items()}
    weight_tensor = torch.FloatTensor([weights[i] for i in sorted(weights.keys())])

    print("Class weights:", {i: f"{w:.2f}" for i, w in enumerate(weight_tensor.numpy())})
    return weight_tensor

def to_device(data, device):
    '''
    Move data to the specified device (CPU or GPU).

    Parameters
    ----------
    data : any
        The data to move to the device.

    device : torch.device
        The device to move the data to.

    Returns
    -------
        any
        The data moved to the specified device.
    '''
    if isinstance(data, torch.Tensor):
        return data.to(device)
    elif isinstance(data, list):
        return [to_device(x, device) for x in data]
    elif isinstance(data, tuple):
        return tuple(to_device(x, device) for x in data)
    else:
        try:
            return data.to(device)
        except:
            print(f"Warning: Could not move data of type {type(data)} to device")
            return data

# # Modified training function to support early stopping and return best validation loss
def train_model(model, train_loader, valid_loader, num_epochs=10, learning_rate=0.001,
               weight_decay=1e-5, patience=5, device="cuda" if torch.cuda.is_available() else "cpu",
               verbose=True, plot=True):
    """
    Train the OCR model with early stopping and hyperparameter optimization support.

    Parameters
    ----------
    model : nn.Module
        The PyTorch model to train.

    train_loader : DataLoader
        DataLoader for the training dataset.

    valid_loader : DataLoader
        DataLoader for the validation dataset.

    num_epochs : int
        Number of epochs to train the model (default: 10).

    learning_rate : float
        Learning rate for the optimizer (default: 0.001).

    weight_decay : float
        Weight decay for the optimizer (default: 1e-5).

    patience : int
        Number of epochs with no improvement after which training will be stopped (default: 5).

    device : str
        Device to use for training (default: "cuda" if available, else "cpu").

    verbose : bool
        Whether to print training progress (default: True).

    plot : bool
        Whether to plot training and validation metrics (default: True).

    Returns
    -------
    model : nn.Module
        The trained model.

    metrics : dict
        Dictionary containing training and validation metrics.

    best_val_loss : float
        The best validation loss achieved during training.

    Examples
    --------
    >>> model = OCRModel(num_classes=2, num_types=5)
    >>> train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    >>> valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False)
    >>> trained_model, metrics, best_val_loss = train_model(model, train_loader, valid_loader)

    """
    if verbose:
        print(f"Using device: {device}")
    model = model.to(device)

    # Debug data structure if verbose
    if verbose:
        print("Examining data structure...")
        for batch in train_loader:
            print(f"Batch type: {type(batch)}")
            print(f"Batch length: {len(batch)}")
            data, labels = batch
            print(f"Data type: {type(data)}")
            if isinstance(data, (list, tuple)):
                print(f"Data elements: {[type(d) for d in data]}")
            print(f"Labels type: {type(labels)}")
            if isinstance(labels, torch.Tensor):
                print(f"Labels shape: {labels.shape}")
            break

    # Check class distribution and visualize if requested
    if verbose:
        print("Analyzing class distribution...")
    class_weights = get_class_weights(train_loader)

    if verbose and plot:
        class_counts = analyze_class_distribution(train_loader)
        if class_counts:
            plt.figure(figsize=(10, 5))
            classes = sorted(class_counts.keys())
            counts = [class_counts[c] for c in classes]
            plt.bar(classes, counts)
            plt.title('Class Distribution in Training Set')
            plt.xlabel('Class')
            plt.ylabel('Count')
            plt.xticks(classes)
            plt.grid(axis='y', linestyle='--', alpha=0.7)
            plt.show()

    # Define weighted loss function and optimizer
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    # Learning rate scheduler
    scheduler = OneCycleLR(optimizer, max_lr=learning_rate, steps_per_epoch=len(train_loader), epochs=num_epochs)

    # Initialize early stopping
    early_stopping = EarlyStopping(patience=patience, verbose=verbose)

    # Lists to store metrics
    metrics = {
        'epoch': [],
        'train_loss': [],
        'val_loss': [],
        'val_accuracy': [],
    }

    best_val_loss = float('inf')
    best_model_state = None

    # Training loop
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        # Use tqdm for progress bar if verbose
        batch_iterator = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}") if verbose else train_loader

        for batch in batch_iterator:
            data, labels = batch
            data = to_device(data, device)
            labels = to_device(labels, device)

            # Forward pass
            outputs = model(data)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()

        # Calculate training loss
        avg_loss = running_loss / len(train_loader) if len(train_loader) > 0 else 0

        # Validation
        model.eval()
        valid_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for batch in valid_loader:
                data, labels = batch
                data = to_device(data, device)
                labels = to_device(labels, device)

                outputs = model(data)
                loss = criterion(outputs, labels)
                valid_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        # Calculate validation metrics
        avg_valid_loss = valid_loss / len(valid_loader) if len(valid_loader) > 0 else 0
        valid_accuracy = 100 * correct / total if total > 0 else 0

        # Store metrics
        metrics['epoch'].append(epoch + 1)
        metrics['train_loss'].append(avg_loss)
        metrics['val_loss'].append(avg_valid_loss)
        metrics['val_accuracy'].append(valid_accuracy)

        # Save best model
        if avg_valid_loss < best_val_loss:
            best_val_loss = avg_valid_loss
            best_model_state = model.state_dict().copy()

        # Print progress
        if verbose:
            print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_loss:.4f}, "
                 f"Val Loss: {avg_valid_loss:.4f}, Accuracy: {valid_accuracy:.2f}%")

        # Early stopping check
        early_stopping(avg_valid_loss)
        if early_stopping.early_stop:
            if verbose:
                print(f"Early stopping triggered at epoch {epoch+1}")
            break

    # Load best model weights
    if best_model_state:
        model.load_state_dict(best_model_state)

    # Create visualization and report if requested
    if verbose and plot:
        # Create a DataFrame and display as a pretty table
        df = pd.DataFrame(metrics)
        print("\n===== Training Results =====")
        print(tabulate(df, headers='keys', tablefmt='pretty', showindex=False))

        # Plot training curves
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 2, 1)
        plt.plot(metrics['epoch'], metrics['train_loss'], 'b-', label='Train Loss')
        plt.plot(metrics['epoch'], metrics['val_loss'], 'r-', label='Val Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.title('Training and Validation Loss')

        plt.subplot(1, 2, 2)
        plt.plot(metrics['epoch'], metrics['val_accuracy'], 'g-')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy (%)')
        plt.title('Validation Accuracy')
        plt.tight_layout()
        plt.show()

    # Return model and metrics for hyperparameter optimization
    return model, metrics, best_val_loss

# Hyperparameter optimization objective function
def objective(trial, model_class, dataset, base_config=None):
    """
    Objective function for Optuna hyperparameter optimization.

    Parameters:
    -----------
    trial : optuna.trial.Trial
        Optuna trial object
    model_class : class
        PyTorch model class to optimize
    dataset : Dataset
        Dataset to use for training and validation
    base_config : dict, optional
        Base configuration to use (default: None)
    """
    # Get model parameter signature to determine what parameters it accepts
    import inspect
    model_params = inspect.signature(model_class.__init__).parameters
    param_names = list(model_params.keys())[1:]  # Skip 'self'

    # Configuration with hyperparameter search space
    config = {
        'learning_rate': trial.suggest_float('learning_rate', 5e-5, 5e-3, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [16, 32, 64]),
        'patience': trial.suggest_int('patience', 3, 10),
        'num_epochs': trial.suggest_int('num_epochs', 20, 50),
    }

    # Add model-specific hyperparameters only if the model accepts them
    if 'dropout1' in param_names:
        config['dropout1'] = trial.suggest_float('dropout1', 0.1, 0.5)
    if 'dropout2' in param_names:
        config['dropout2'] = trial.suggest_float('dropout2', 0.1, 0.5)

    # If any additional base config is provided
    if base_config:
        for k, v in base_config.items():
            if k not in config:
                config[k] = v

    # Create data loaders with trial-specific batch size
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, valid_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=config['batch_size'])

    # Filter kwargs to only include parameters the model accepts
    model_kwargs = {k: v for k, v in config.items() if k in param_names}
    model_kwargs['num_classes'] = len(dataset.label_mapping)
    model_kwargs['num_types'] = len(dataset.type_mapping)

    # Create model with trial-specific parameters
    model = model_class(**model_kwargs)

    # Train model with early stopping
    _, _, best_val_loss = train_model(
        model=model,
        train_loader=train_loader,
        valid_loader=valid_loader,
        learning_rate=config['learning_rate'],
        weight_decay=config['weight_decay'],
        num_epochs=config['num_epochs'],
        patience=config['patience'],
        verbose=False,
        plot=False
    )

    return best_val_loss

# Function to run hyperparameter optimization
def run_hyperparameter_search(model_class, dataset, n_trials=50):
    """
    Run hyperparameter optimization using Optuna.

    Parameters:
    -----------
    model_class : class
        The PyTorch model class to optimize
    dataset : torch.utils.data.Dataset
        The full dataset to use
    n_trials : int
        Number of trials for Optuna

    Returns:
    --------
    study : optuna.study.Study
        The completed Optuna study object
    best_params : dict
        The best parameters found
    """
    print("Starting hyperparameter optimization...")

    # Define the objective function wrapper
    def objective_wrapper(trial):
        return objective(trial, model_class, dataset)

    # Create Optuna study
    study = optuna.create_study(direction="minimize")

    try:
        study.optimize(objective_wrapper, n_trials=n_trials)
    except KeyboardInterrupt:
        print("Optimization interrupted by user.")

    print("Hyperparameter optimization completed!")
    print(f"Best validation loss: {study.best_value:.4f}")
    print("Best hyperparameters:", study.best_params)

    return study, study.best_params


# Run hyperparameter optimization
study, best_params = run_hyperparameter_search(OCRModel, dataset, n_trials=20)

# Create data loaders with the best batch size
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, valid_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=best_params['batch_size'], shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=best_params['batch_size'])

# Create and train the best model
best_model = OCRModel(
    num_classes=len(dataset.label_mapping),
    num_types=len(dataset.type_mapping),
    dropout1=best_params['dropout1'],
    dropout2=best_params['dropout2']
)

trained_model, metrics, _ = train_model(
    best_model,
    train_loader,
    valid_loader,
    num_epochs=best_params['num_epochs'],
    learning_rate=best_params['learning_rate'],
    weight_decay=best_params['weight_decay'],
    patience=best_params['patience']
)

# Save the best model
torch.save(trained_model.state_dict(), 'best_ocr_model.pth')


In [29]:
# Set random seeds
set_seeds(42)

# Create model with the best hyperparameters
model = OCRModel(
    num_classes=len(dataset.label_mapping),
    num_types=len(dataset.type_mapping),
    dropout1=0.30809226632994047,
    dropout2=0.29020378370535227
)

# Setup data loaders using best batch size
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, valid_dataset = random_split(dataset, [train_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=16)

# Define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on {device}")

# Train the model with the best hyperparameters for 25 epochs
model, metrics, best_val_loss = train_model(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    num_epochs=25,  # Train for 25 epochs as per best hyperparameters
    learning_rate=0.0009008692431483686,
    weight_decay=5.314580211395843e-06,
    patience=9,     # Early stopping patience
    device=device,
    verbose=True,   # Show progress
    plot=True       # Generate plots
)

# Save the trained model
torch.save(model.state_dict(), 'ocr_model_25epochs.pth')

# Show best validation loss achieved
print(f"\nBest validation loss: {best_val_loss:.4f}")

# Check if we have accuracy in the metrics
if 'val_accuracy' in metrics:
    best_acc_idx = metrics['val_accuracy'].index(max(metrics['val_accuracy']))
    best_acc = metrics['val_accuracy'][best_acc_idx]
    best_acc_epoch = metrics['epoch'][best_acc_idx]
    print(f"Best accuracy: {best_acc:.2f}% (epoch {best_acc_epoch})")

# Evaluate on test set if available
if 'test_loader' in globals():
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in test_loader:
            data, labels = batch
            data = to_device(data, device)
            labels = to_device(labels, device)

            outputs = model(data)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    test_accuracy = 100 * correct / total
    print(f"Test accuracy: {test_accuracy:.2f}%")

In [ ]:
# Simpler classifier for OCR
class OCRModel(nn.Module):
    def __init__(self):
        super(OCRModel, self).__init__()
        # Image processing path
        self.image_layer = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),  # Convolutional layer with 16 filters, 3x3 kernel
            nn.MaxPool2d(kernel_size=2),  # Max pooling layer with 2x2 kernel to reduce spatial dimensions
            nn.ReLU(),  # ReLU activation function
            nn.Flatten(),  # Flatten the output to feed into a fully connected layer
            nn.Linear(16*32*32, 128)  # Fully connected layer to reduce dimensionality to 128
        )

        # Type processing path
        self.type_layer = nn.Sequential(
            nn.Linear(5, 10),  # Fully connected layer to process the type vector
            nn.ReLU(),  # ReLU activation function
        )

        # Combined classifier
        self.classifier = nn.Sequential(
            nn.Linear(128 + 10, 64),  # Fully connected layer to combine image and type features
            nn.ReLU(),  # ReLU activation function
            nn.Linear(64, 2)  # Output layer with 2 classes (primary ID, secondary ID)
        )

    def forward(self, x_image, x_type):
        # Forward pass for image processing
        x_image = self.image_layer(x_image)

        # Forward pass for type processing
        x_type = self.type_layer(x_type)

        # Concatenate image and type features
        x = torch.cat((x_image, x_type), dim=1)

        # Final classification
        return self.classifier(x)

# Load the data in batches
train_dataloader = DataLoader(dataset, batch_size=10, shuffle=True)

# Call the model
model = OCRModel()

# Define the optimizer and loss function
# originally Adam optimizer: AdamW is a variant of Adam with weight decay
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Train for ten epochs
# More epochs may be needed for convergence
for epoch in range(10):
    for (images, types), labels in train_dataloader:
        optimizer.zero_grad()
        outputs = model(images, types)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")


## Conclusion:


In this notebook, we have developed a multi-modal model for Optical Character Recognition (OCR) tasks. The model takes both image and text as input and uses a convolutional neural network (CNN) to extract features from the images. The text input is processed through an embedding layer to convert it into a numerical representation. The two outputs are then concatenated and passed through a fully connected layer to produce the final output.

The hyperparameter tuning with Optuna was successful, and we were able to find the best hyperparameters for the model. The model was trained on a small dataset of 100 samples, and the results were promising. The model was able to achieve a validation accuracy of 0.65, which is a good starting point for further improvements. However, the accuracy shoots upto 90% therefore, residual blocks were added to the model. The residual blocks help the model learn better by allowing it to skip certain layers and focus on the important features. This is especially useful in deep networks where the gradients can vanish or explode, making it difficult for the model to learn. The residual blocks help mitigate this issue and improve the overall performance of the model.